## Visualization Notebook
Load pre-computed results from the pipeline notebook and generate all figures.

In [1]:
import matplotlib
matplotlib.use('Agg')
#Imports
from glob import glob
import numpy as np
import pandas as pd
import os
from matplotlib import pyplot as plt
import cv2
from scipy import stats
from scipy.ndimage import uniform_filter1d, gaussian_filter1d
from scipy.ndimage import gaussian_filter
from scipy import signal
import seaborn as sns

%run loom_analysis_functions.py

machine = 'hilbert'

if machine == 'adams':
    PROJECT_PATH = '/ssd02/Projects/Data/jen_projects/virtualcricket_loom/'
elif machine == 'hilbert':
    PROJECT_PATH = '/media/arnab/My Book Duo/Data/hoylab_projects/virtualcricket_loom/'

save_dir = os.path.join(PROJECT_PATH, 'analysis_results')

SVG_DIR = os.path.join(os.getcwd(), 'figure_svgs')
os.makedirs(SVG_DIR, exist_ok=True)

In [2]:
# Load all saved results
results = np.load(os.path.join(save_dir, 'results.npz'), allow_pickle=True)['results'].item()
blender_data = np.load(os.path.join(save_dir, 'blender_data.npz'), allow_pickle=True)['blender_data'].item()
mouse_stats = np.load(os.path.join(save_dir, 'mouse_stats.npz'), allow_pickle=True)['mouse_stats'].item()
mouse_stats_aggregated = np.load(os.path.join(save_dir, 'mouse_stats_aggregated.npz'), allow_pickle=True)['mouse_stats_aggregated'].item()
ethogram_data = np.load(os.path.join(save_dir, 'ethogram_data.npz'), allow_pickle=True)['ethogram_data'].item()
escape_freeze = np.load(os.path.join(save_dir, 'escape_arrays.npz'), allow_pickle=True)
escape_arrays_by_group = escape_freeze['escape_arrays_by_group'].item()
freeze_arrays_by_group = escape_freeze['freeze_arrays_by_group'].item()
meta = np.load(os.path.join(save_dir, 'metadata.npz'), allow_pickle=True)
group_names = list(meta['group_names'])
mice_id = meta['mice_id'].item()
analysis_params = meta['analysis_params'].item()

# Load looming timestamps (needed by some plots)
looming_timestamps = dict(np.load(os.path.join(PROJECT_PATH, 'looming_timestamps.npz'), allow_pickle=True))

print(f"Loaded {len(results)} groups")
for gn in group_names:
    print(f"  {gn}: {len(results[gn])} videos")
print(f"\nAnalysis time window: {analysis_params['max_seconds_from_first_loom']}s from first loom")

Loaded 8 groups
  naive_adult_males_shelter: 27 videos
  naive_adult_females_shelter: 30 videos
  naive_adolescent_males_shelter: 27 videos
  naive_adolescent_females_shelter: 21 videos
  experienced_adult_males_shelter: 24 videos
  experienced_adult_females_shelter: 24 videos
  experienced_adolescent_males_shelter: 24 videos
  experienced_adolescent_females_shelter: 24 videos

Analysis time window: 120s from first loom


## Plot Settings

In [3]:
pad_onset = analysis_params['pad_onset']
pad_offset = analysis_params['pad_offset']
fps = analysis_params['fps']
loom_duration = analysis_params['loom_duration']
escape_thresholds = analysis_params['escape_thresholds']
speeds = analysis_params['speeds']
max_seconds_from_first_loom = analysis_params['max_seconds_from_first_loom']

## Shelter Distance Trajectories

In [4]:
escape_distance = 50
font_size = 8

experience = 'naive'
to_plot = 'escape'

if to_plot == 'escape':
    colors = {}
    colors['adult_female'] = 'purple' #[124/255, 156/255, 227/255]
    colors['adult_male'] =  'gray' #[138/255, 138/255, 138/255]
    colors['adolescent_female'] = 'purple' #[124/255, 156/255, 227/255]
    colors['adolescent_male'] = 'gray' #[138/255, 138/255, 138/255]
else:
    colors = {}
    colors['adult_female'] = [51/255, 157/255, 157/255]
    colors['adult_male'] = [138/255, 138/255, 138/255]
    colors['adolescent_female'] = [51/255, 157/255, 157/255]
    colors['adolescent_male'] = [138/255, 138/255, 138/255]

fig, axes = plt.subplots(1, 2, figsize=(4.5, 2.5), dpi=300, sharey=True)

ages = ['adolescent', 'adult']

for ax_idx, age in enumerate(ages):
    ax = axes[ax_idx]
    
    # Create empty plots for legend
    female_line = ax.plot([], [], color='red', alpha=0.8, label='Females')[0]
    male_line = ax.plot([], [], color='blue', alpha=0.8, label='Males')[0]
    threshold_line = ax.plot([], [], color='black', linestyle='--', label='Escape threshold')[0]

    # Store data for mean calculation
    female_distances = []
    male_distances = []

    for group_name in results.keys():
        for video_filename in results[group_name].keys():
            vid_pix2cm = results[group_name][video_filename]['pix2cm_scale']
            for mouse_shelter_dist, escape, mouse_spd, cricket_dist, freeze in zip(results[group_name][video_filename]['mouse_shelter_distance_trialwise'],
                                                results[group_name][video_filename]['escape_to_shelter_trialwise'],
                                                results[group_name][video_filename]['mouse_speed_trialwise'],
                                                results[group_name][video_filename]['mouse_cricket_distance_trialwise'],
                                                results[group_name][video_filename]['freeze_trialwise']):
                if to_plot == 'freeze':
                    if freeze:
                        #aggregate all trials for a mouse
                        if 'female' in group_name and age in group_name and experience in group_name:
                            ax.plot(np.arange(len(mouse_shelter_dist))/30, (mouse_shelter_dist/vid_pix2cm) - 5, color = colors[f'{age}_female'], alpha = 0.2)
                            female_distances.append(mouse_shelter_dist/vid_pix2cm)
                        elif 'female' not in group_name and age in group_name and experience in group_name:
                            ax.plot(np.arange(len(mouse_shelter_dist))/30, (mouse_shelter_dist/vid_pix2cm) - 5, color = colors[f'{age}_male'], alpha = 0.2)
                            male_distances.append(mouse_shelter_dist/vid_pix2cm)
                elif to_plot == 'escape':
                    if escape:
                        if 'female' in group_name and age in group_name and experience in group_name:
                            ax.plot(np.arange(len(mouse_shelter_dist))/30, (mouse_shelter_dist/vid_pix2cm) - 5, color = colors[f'{age}_female'], alpha = 0.2)
                            female_distances.append(mouse_shelter_dist/vid_pix2cm)
                        elif 'female' not in group_name and age in group_name and experience in group_name:
                            ax.plot(np.arange(len(mouse_shelter_dist))/30, (mouse_shelter_dist/vid_pix2cm) - 5, color = colors[f'{age}_male'], alpha = 0.2)
                            male_distances.append(mouse_shelter_dist/vid_pix2cm)

    # Calculate and plot mean of individual and mean of mean lines
    if female_distances:
        # Find the maximum length to pad shorter arrays
        max_len = max(len(dist) for dist in female_distances)
        # Pad arrays with NaN and calculate mean ignoring NaN
        padded_female = np.array([np.pad(dist, (0, max_len - len(dist)), constant_values=np.nan) for dist in female_distances])
        female_mean = np.nanmean(padded_female, axis=0)
        
        # Smooth the mean line using a moving average
        from scipy.ndimage import uniform_filter1d
        female_mean_smooth = uniform_filter1d(female_mean, size=5, mode='nearest')
        ax.plot(np.arange(len(female_mean_smooth))/30, female_mean_smooth, color=colors[f'{age}_female'],
         linewidth=2, alpha=1)

    if male_distances:
        # Find the maximum length to pad shorter arrays
        max_len = max(len(dist) for dist in male_distances)
        # Pad arrays with NaN and calculate mean ignoring NaN
        padded_male = np.array([np.pad(dist, (0, max_len - len(dist)), constant_values=np.nan) for dist in male_distances])
        male_mean = np.nanmean(padded_male, axis=0)
        
        # Smooth the mean line using a moving average
        from scipy.ndimage import uniform_filter1d
        male_mean_smooth = uniform_filter1d(male_mean, size=5, mode='nearest')
        ax.plot(np.arange(len(male_mean_smooth))/30, male_mean_smooth, color=colors[f'{age}_male'], linewidth=2, alpha=1)

    ax.hlines(50/vid_pix2cm, 0, 4.3, color = 'black', linestyle = '--')
    ax.set_xlabel('Time (s)', fontsize=font_size)
    if age == 'adolescent':
        ax.set_ylabel('Distance to shelter (cm)', fontsize=font_size)
    ax.set_title(f'{age}', fontsize=font_size)
    ax.tick_params(axis='both', labelsize=font_size)
    ax.axvspan(1, 2.5, color = 'gray', alpha = 0.1, linewidth=0)
    ax.set_ylim(-5,68)

plt.tight_layout()
plt.show()
fig.savefig(os.path.join(SVG_DIR, f'{experience}_{to_plot}_shelter_distances_comparison.svg'))
#kmeans on mouse to shelter distance or mouse to cricket distance or both? this below can help parse escape and non escape trials and continued pursuit trials
#why are some trials longer than set duration?

/tmp/ipykernel_16401/2673879112.py:81: RuntimeWarning: Mean of empty slice
  male_mean = np.nanmean(padded_male, axis=0)
/tmp/ipykernel_16401/2673879112.py:98: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Cricket Distance (All Trials)

In [5]:
escape_distance = 50

plt.figure(figsize = (5, 5))
for group_name in results.keys():
    for video_filename in results[group_name].keys():
        for mouse_shelter_dist, escape, mouse_spd, cricket_dist in zip(results[group_name][video_filename]['mouse_shelter_distance_trialwise'],
                                            results[group_name][video_filename]['escape_to_shelter_trialwise'],
                                            results[group_name][video_filename]['mouse_speed_trialwise'],
                                            results[group_name][video_filename]['mouse_cricket_distance_trialwise']):
            # if escape:
            #     if 'adult' in group_name:
            #         plt.plot(cricket_dist, color = 'red', alpha = 0.2)
            #     else:
            #         plt.plot(cricket_dist, color = 'blue', alpha = 0.2)
            # else:
                plt.plot(cricket_dist, '.', color = 'black', alpha = 0.01)

plt.hlines(50, 0, 140, color = 'black', linestyle = '--')

#kmeans on mouse to shelter distance or mouse to cricket distance or both? this below can help parse escape and non escape trials and continued pursuit trials
#why are some trials longer than set duration?
plt.savefig(os.path.join(SVG_DIR, 'cricket_distance_all_trials.svg'), format='svg', bbox_inches='tight')

## Escape Speed Analysis

In [6]:
escape_distance = 50

plt.figure(figsize = (5, 5))
for group_name in results.keys():
    for video_filename in results[group_name].keys():
        for mouse_shelter_dist, escape, mouse_speed in zip(results[group_name][video_filename]['mouse_shelter_distance_trialwise'],
                                            results[group_name][video_filename]['escape_to_shelter_trialwise'],
                                            results[group_name][video_filename]['mouse_speed_trialwise']):
            if escape:
                shelter_indices = np.where(np.array(mouse_shelter_dist) < escape_distance)[0]
                if len(shelter_indices) == 0:
                    continue
                reached_shelter_idx = shelter_indices[0]

                speed = gaussian_filter(mouse_speed[pad_onset:reached_shelter_idx], 2)
                if len(speed) == 0:
                    continue
                std = np.nanstd(speed)
                n_stds = 1
                
                peaks = signal.find_peaks(speed, std * n_stds, width=1)[0]
                if len(peaks) == 0:
                    continue
                all_peak_starts = peaks[0]

                plt.scatter(np.arange(len(mouse_shelter_dist[pad_onset:reached_shelter_idx])),
                    mouse_shelter_dist[pad_onset:reached_shelter_idx], 
                    c = mouse_speed[pad_onset:reached_shelter_idx], vmin = 0, vmax = 50)
                plt.scatter(np.arange(len(mouse_shelter_dist[pad_onset+all_peak_starts:reached_shelter_idx]))+all_peak_starts,
                    mouse_shelter_dist[pad_onset+all_peak_starts:reached_shelter_idx], edgecolor='red', facecolor='none', alpha = 0.5, s=100)
                plt.show()
            # else:
            #     plt.plot(gaussian_filter(mouse_shelter_dist, 2), color = 'blue', alpha = 0.2)

plt.hlines(50, 0, 140, color = 'black', linestyle = '--')

/tmp/ipykernel_16401/3486809558.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Escape Probability Over Trials

In [7]:
plt.rcParams.update({'font.size': 8})
male_color = '#808080'  # Dark gray
female_color = 'purple'  # Steel blue

fig, axs = plt.subplots(2, 1, figsize = (2.5, 2.5), dpi = 300, sharex=True)
axs = axs.ravel()
var = 'experienced'
for gp in escape_arrays_by_group.keys():
    if var in gp:
        if 'adolescent' in gp:
            if 'female' in gp:
                color = female_color
            else:
                color = male_color
            axs[0].errorbar(np.arange(escape_arrays_by_group[gp].shape[1]),
                        np.nanmean(escape_arrays_by_group[gp], axis=0), 
                        np.nanstd(escape_arrays_by_group[gp], axis=0)/np.sqrt(np.sum(~np.isnan(escape_arrays_by_group[gp]), axis=0)),
                        label=gp, linestyle='-', linewidth=1, alpha = 0.7,
                        marker = 'o', markersize = 1, capsize = 1, color = color)
            axs[0].set_title('adolescent')
            axs[0].set_xticks(np.arange(10))
            axs[0].set_xticklabels(np.arange(1, 11))
            axs[0].set_yticks([0, 0.5, 1])
            axs[0].spines['top'].set_visible(False)
            axs[0].spines['right'].set_visible(False)
            axs[0].grid(True, axis='y', alpha=0.2)

        else:
            if 'female' in gp:
                color = female_color
            else:
                color = male_color
            axs[1].errorbar(np.arange(escape_arrays_by_group[gp].shape[1]),
                        np.nanmean(escape_arrays_by_group[gp], axis=0),
                        np.nanstd(escape_arrays_by_group[gp], axis=0)/np.sqrt(np.sum(~np.isnan(escape_arrays_by_group[gp]), axis=0)),
                        label=gp, linestyle='-', linewidth=1, alpha = 0.7,
                        marker = 'o', markersize = 1, capsize = 1, color = color)
            axs[1].set_title('adult')
            axs[1].set_xticks(np.arange(10))
            axs[1].set_xticklabels(np.arange(1, 11))
            axs[1].set_yticks([0, 0.5, 1])
            axs[1].spines['top'].set_visible(False)
            axs[1].spines['right'].set_visible(False)
            axs[1].grid(True, axis='y', alpha=0.2)
            axs[1].set_ylabel('Escape\n probability')
            axs[0].set_ylabel('Escape\n probability')
            axs[1].set_xlabel('Trial number')
plt.tight_layout()
plt.savefig(os.path.join(SVG_DIR, f'{var}_escape_array_plot.svg'), dpi=300, bbox_inches='tight')

/tmp/ipykernel_16401/1918355809.py:34: RuntimeWarning: Mean of empty slice
  np.nanmean(escape_arrays_by_group[gp], axis=0),
/home/arnab/miniforge3/envs/analysis/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_16401/1918355809.py:34: RuntimeWarning: Mean of empty slice
  np.nanmean(escape_arrays_by_group[gp], axis=0),
/home/arnab/miniforge3/envs/analysis/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_16401/1918355809.py:16: RuntimeWarning: Mean of empty slice
  np.nanmean(escape_arrays_by_group[gp], axis=0),
/home/arnab/miniforge3/envs/analysis/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=d

## Per-Mouse Metrics (Half-Violin Plots)

In [8]:
# Calculate global y-axis limits for consistent scaling
metrics = ['distance_travelled', 'time_spent_in_shelter', 'number_of_approaches']
metric_labels = ['Distance Travelled (cm)', 'Time Spent in Shelter (s)', 'Number of Approaches']

global_ylims = {}
for metric in metrics:
    all_values = []
    for group_name in group_names:
        if group_name in mouse_stats_aggregated and metric in mouse_stats_aggregated[group_name]:
            all_values.extend(mouse_stats_aggregated[group_name][metric])
    if all_values:
        global_ylims[metric] = (0, max(all_values) * 1.1)
    else:
        global_ylims[metric] = (0, 1)

In [9]:
def create_half_violin(ax, data, positions, colors, widths=0.3, side='left', alpha=0.6):
    """Create half violin plots"""
    from matplotlib.patches import Polygon
    
    for i, (d, pos) in enumerate(zip(data, positions)):
        if len(d) > 0:
            # Calculate kernel density
            kde = stats.gaussian_kde(d)
            # Create points for evaluation
            y_min, y_max = np.min(d) - np.std(d), np.max(d) + np.std(d)
            y_points = np.linspace(y_min, y_max, 100)
            density = kde(y_points)
            
            # Normalize density for width
            density = density / np.max(density) * widths
            
            # Create half violin
            if side == 'left':
                x_points = pos - density
            else:
                x_points = pos + density
            
            # Create polygon for filling
            if side == 'left':
                vertices = list(zip(x_points, y_points)) + [(pos, y_points[-1]), (pos, y_points[0])]
            else:
                vertices = [(pos, y_points[0]), (pos, y_points[-1])] + list(zip(x_points, y_points))[::-1]
            
            poly = Polygon(vertices, facecolor=colors[i], alpha=alpha, edgecolor='none')
            ax.add_patch(poly)

def add_jittered_points(ax, data, positions, colors, jitter=0.05, alpha=0.7, size=50):
    """Add jittered scatter points with circular markers"""
    for i, (d, pos) in enumerate(zip(data, positions)):
        if len(d) > 0:
            # Add jitter
            x_jitter = np.random.normal(0, jitter, size=len(d))
            x_positions = pos + x_jitter
            
            # Plot points with circular markers
            ax.scatter(x_positions, d, color=colors[i], s=size, 
                      edgecolors='white', linewidth=0.8, zorder=10, marker='o')

def add_mean_bars(ax, data, positions, width=0.2):
    """Add mean indicator bars"""
    for i, (d, pos) in enumerate(zip(data, positions)):
        if len(d) > 0:
            mean_val = np.median(d)
            ax.plot([pos - width/2, pos + width/2], [mean_val, mean_val], 
                   color='black', linewidth=2.5, zorder=15)

# Simulate example data (replace this with your actual mouse_stats_aggregated)
np.random.seed(42)

# Define colors matching the reference image
male_color = '#C0C0C0'  # Light gray
female_color = '#ADD8E6'  # Light blue

# Create individual plots for each metric
metrics = ['distance_travelled', 'time_spent_in_shelter', 'number_of_approaches']
metric_labels = ['Distance Travelled (cm)', 'Time Spent in Shelter (s)', 'Number of Approaches']

# Calculate global min and max for each metric to ensure consistent y-axis scaling
for metric, metric_label in zip(metrics, metric_labels):
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    fig.suptitle(f'{metric_label}', fontsize=16, fontweight='bold')
    
    experience_levels = ['naive', 'experienced']
    experience_labels = ['Naive', 'Experienced']
    
    for col, (experience, exp_label) in enumerate(zip(experience_levels, experience_labels)):
        ax = axes[col]
        
        # Prepare data for plotting
        adolescent_male_data = mouse_stats_aggregated[f'{experience}_adolescent_males_shelter'][metric]
        adolescent_female_data = mouse_stats_aggregated[f'{experience}_adolescent_females_shelter'][metric]
        adult_male_data = mouse_stats_aggregated[f'{experience}_adult_males_shelter'][metric]
        adult_female_data = mouse_stats_aggregated[f'{experience}_adult_females_shelter'][metric]
        
        # Set positions - grouped by age, separated by sex
        adolescent_positions = [1, 1.4]  # male, female
        adult_positions = [2.6, 3]  # male, female
        
        # Create half violins for adolescents
        create_half_violin(ax, [adolescent_male_data], [adolescent_positions[0]], 
                          [male_color], widths=0.35, side='left', alpha=0.5)
        create_half_violin(ax, [adolescent_female_data], [adolescent_positions[1]], 
                          [female_color], widths=0.35, side='right', alpha=0.5)
        
        # Create half violins for adults
        create_half_violin(ax, [adult_male_data], [adult_positions[0]], 
                          [male_color], widths=0.35, side='left', alpha=0.5)
        create_half_violin(ax, [adult_female_data], [adult_positions[1]], 
                          [female_color], widths=0.35, side='right', alpha=0.5)
        
        # Add jittered points
        add_jittered_points(ax, [adolescent_male_data], [adolescent_positions[0]], 
                           [male_color], jitter=0.04, alpha=0.6)
        add_jittered_points(ax, [adolescent_female_data], [adolescent_positions[1]], 
                           [female_color], jitter=0.04, alpha=0.6)
        add_jittered_points(ax, [adult_male_data], [adult_positions[0]], 
                           [male_color], jitter=0.04, alpha=0.6)
        add_jittered_points(ax, [adult_female_data], [adult_positions[1]], 
                           [female_color], jitter=0.04, alpha=0.6)
        
        # Add mean bars
        add_mean_bars(ax, [adolescent_male_data], [adolescent_positions[0]], width=0.15)
        add_mean_bars(ax, [adolescent_female_data], [adolescent_positions[1]], width=0.15)
        add_mean_bars(ax, [adult_male_data], [adult_positions[0]], width=0.15)
        add_mean_bars(ax, [adult_female_data], [adult_positions[1]], width=0.15)
        
        # Styling
        ax.set_xticks([1.2, 2.8])
        ax.set_xticklabels(['Adolescent', 'Adult'], fontsize=14)
        
        # # Add secondary x-axis labels for sex
        # ax.text(1, ax.get_ylim()[0] - (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.12, 
        #         'male', ha='center', fontsize=11)
        # ax.text(1.4, ax.get_ylim()[0] - (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.12, 
        #         'female', ha='center', fontsize=11)
        # ax.text(2.6, ax.get_ylim()[0] - (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.12, 
        #         'male', ha='center', fontsize=11)
        # ax.text(3, ax.get_ylim()[0] - (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.12, 
        #         'female', ha='center', fontsize=11)
        
        ax.set_title(exp_label, fontsize=14, fontweight='bold')
        
        if col == 0:
            ax.set_ylabel(metric_label, fontsize=12)
        
        # Add grid
        # ax.yaxis.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
        ax.set_axisbelow(True)
        
        # Remove top and right spines
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        # Set consistent y-axis limits across both subplots
        ax.set_ylim(global_ylims[metric])
    
    plt.tight_layout()
    
    # Save individual plots
    filename = os.path.join(SVG_DIR, f'{metric}_plot.png')
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename}")
    
plt.show()

print("\n✓ All plots generated successfully!")
print("\nTo use with your actual data:")
print("1. Replace the sample data generation with your actual mouse_stats_aggregated")
print("2. The plots will automatically adapt to your data")
print("3. Adjust colors, sizes, and other parameters as needed")

Saved: /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/distance_travelled_plot.png


Saved: /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/time_spent_in_shelter_plot.png


Saved: /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/number_of_approaches_plot.png

✓ All plots generated successfully!

To use with your actual data:
1. Replace the sample data generation with your actual mouse_stats_aggregated
2. The plots will automatically adapt to your data
3. Adjust colors, sizes, and other parameters as needed


/tmp/ipykernel_16401/979800045.py:149: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Per-Mouse Metrics (Seaborn Violin Plots)

In [10]:
experience_levels = ['naive', 'experienced']

for experience in experience_levels:
    for metric, metric_label in zip(metrics, metric_labels):
        fig, axes = plt.subplots(1, 2, figsize=(4.5, 2.5), dpi=300)
        # fig.suptitle(f'{metric_label} - {experience.capitalize()}', fontsize=8, fontweight='bold')
        
        age_groups = ['adolescent', 'adult']
        age_labels = ['Adolescent', 'Adult']
        
        for col, (age_group, age_label) in enumerate(zip(age_groups, age_labels)):
            ax = axes[col]
            
            # Prepare data for plotting
            male_data = mouse_stats_aggregated[f'{experience}_{age_group}_males_shelter'][metric]
            female_data = mouse_stats_aggregated[f'{experience}_{age_group}_females_shelter'][metric]
            
            # Create DataFrame for seaborn
            data_list = []
            for val in male_data:
                data_list.append({'sex': 'male', 'value': val})
            for val in female_data:
                data_list.append({'sex': 'female', 'value': val})
            
            plot_df = pd.DataFrame(data_list)
            
            # Define sex order and palette
            sex_order = ['male', 'female']
            sex_palette = {'male': male_color, 'female': female_color}
            
            # Create seaborn violin plots
            sns.violinplot(data=plot_df, x='sex', y='value', order=sex_order, 
                          palette=sex_palette, alpha=0.3, inner=None, split=True, width=0.25, ax=ax)
            
            # Plot individual points with jitter
            for i, sex_val in enumerate(sex_order):
                sex_data = plot_df[plot_df['sex'] == sex_val]['value'].dropna()
                if len(sex_data) > 0:
                    x_jitter = np.random.normal(i, 0.04, len(sex_data))
                    ax.scatter(x_jitter, sex_data, color=sex_palette[sex_val], alpha=0.8, s=15, 
                              facecolors='none', edgecolors=sex_palette[sex_val])
                    
                    # Plot mean lines with error bars
                    mean_val = np.mean(sex_data)
                    sem_val = np.std(sex_data) / np.sqrt(len(sex_data))
                    
                    ax.hlines(mean_val, i-0.1, i+0.1, color='black', linewidth=1)
                    ax.errorbar(i, mean_val, yerr=sem_val, color='black', capsize=2.2, capthick=1.2, linewidth=1.2)
            
            # Styling
            # ax.set_xlabel('Sex', fontsize=8)
            if col == 0:
                ax.set_ylabel(metric_label, fontsize=8)
            else:
                ax.set_ylabel('')
            
            ax.set_title(age_label, fontsize=8, fontweight='bold')
            
            # Remove top and right spines (despine)
            sns.despine(ax=ax)
            
            # Set consistent y-axis limits across both subplots
            if metric in global_ylims:
                # Set ylim to start just above 0 but ensure ticks start at 0
                current_ylim = global_ylims[metric]
                ax.set_ylim(0, current_ylim[1])  # Start just above 0
                # Force y-axis ticks to start at 0
                yticks = ax.get_yticks()
                if yticks[0] > 0:
                    yticks = np.concatenate([[0], yticks])
                ax.set_yticks(yticks)
            
            ax.tick_params(axis='both', which='major', labelsize=8)
        
        plt.tight_layout()
        
        # Save individual plots
        filename = os.path.join(SVG_DIR, f'{experience}_{metric}_plot.png')
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"Saved: {filename}")
        
        # Also save as SVG
        svg_filename = os.path.join(SVG_DIR, f'{experience}_{metric}_plot.svg')
        plt.savefig(svg_filename, format='svg', dpi=300, bbox_inches='tight')
        print(f"Saved: {svg_filename}")
        
        plt.show()

print("\n✓ All plots generated successfully!")

/tmp/ipykernel_16401/841617409.py:32: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_df, x='sex', y='value', order=sex_order,
/tmp/ipykernel_16401/841617409.py:32: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_df, x='sex', y='value', order=sex_order,


Saved: /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/naive_distance_travelled_plot.png
Saved: /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/naive_distance_travelled_plot.svg


/tmp/ipykernel_16401/841617409.py:87: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16401/841617409.py:32: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_df, x='sex', y='value', order=sex_order,
/tmp/ipykernel_16401/841617409.py:32: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_df, x='sex', y='value', order=sex_order,


Saved: /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/naive_time_spent_in_shelter_plot.png
Saved: /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/naive_time_spent_in_shelter_plot.svg


/tmp/ipykernel_16401/841617409.py:87: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16401/841617409.py:32: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_df, x='sex', y='value', order=sex_order,
/tmp/ipykernel_16401/841617409.py:32: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_df, x='sex', y='value', order=sex_order,


...

/tmp/ipykernel_16401/841617409.py:87: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16401/841617409.py:32: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_df, x='sex', y='value', order=sex_order,
/tmp/ipykernel_16401/841617409.py:32: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_df, x='sex', y='value', order=sex_order,


Saved: /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/experienced_time_spent_in_shelter_plot.png
Saved: /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/experienced_time_spent_in_shelter_plot.svg


/tmp/ipykernel_16401/841617409.py:87: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16401/841617409.py:32: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_df, x='sex', y='value', order=sex_order,
/tmp/ipykernel_16401/841617409.py:32: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_df, x='sex', y='value', order=sex_order,


Saved: /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/experienced_number_of_approaches_plot.png
Saved: /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/experienced_number_of_approaches_plot.svg

✓ All plots generated successfully!


/tmp/ipykernel_16401/841617409.py:87: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Ethogram Plots

In [11]:
line_thickness = 3
line_length = 0.7
escape_color = '#478900ff'
no_response_color = 'white'
freeze_color = '#7babd5ff'
background_color = 'lightgray'

for experience in ['naive', 'experienced']:
    fig, axs = plt.subplots(2, 2, figsize = (4.5, 2.5), dpi = 300, sharex=True)
    axs = axs.ravel()
    count = 0
    for age in ['adolescent', 'adult']:
        for sex in ['males', 'females']:
            ax = axs[count]
            
            # Add background color using axvspan
            ax.axvspan(0, 27000, alpha=0.1, color=background_color, zorder=-2000, linewidth=0)
            
            ax.eventplot(ethogram_data[f'{experience}_{age}_{sex}_shelter']['escape_times'], colors = escape_color, linewidths=line_thickness,
             linelengths=line_length, zorder = 1000, linestyles='solid', alpha = 0.9)
            
            # Use scatter for no_response_times with square markers
            no_response_times = ethogram_data[f'{experience}_{age}_{sex}_shelter']['no_response_times']
            for i, times in enumerate(no_response_times):
                if len(times) > 0:
                    ax.scatter(times, [i] * len(times), c=no_response_color, s=9.8, marker='s', 
                              edgecolors='black', linewidths=0.2, alpha=0.7, zorder=-500)
            
            ax.eventplot(ethogram_data[f'{experience}_{age}_{sex}_shelter']['freeze_times'], colors = freeze_color, linewidths=line_thickness,
                linelengths=line_length, alpha = 0.9)

            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.grid(True, axis='y', alpha=0.2)
            # Set y-axis ticks to integers starting at 1 with spacing of 1
            num_mice = len(ethogram_data[f'{experience}_{age}_{sex}_shelter']['escape_times'])
            ax.set_yticks(range(0, num_mice))
            ax.set_yticklabels(range(1, num_mice + 1), fontsize=3.5)
            ax.set_title(f'{age} {sex}', fontsize=8)

            tick_positions = np.arange(0, 27001, 6000)  # 3000 frames = 100 seconds at 30 fps
            tick_labels = [f"{tick/30:.0f}" for tick in tick_positions]
            ax.set_xticks(tick_positions, tick_labels, fontsize=8)
            if count >1:
                ax.set_xlabel('Time (s)', fontsize=8)
            ax.axvspan(300 *30, 600 *30, alpha=0.23, color='gray', zorder = -1000, linewidth=0)
            ax.set_xlim(0, 27000)
            count += 1

    plt.tight_layout()
    plt.savefig(os.path.join(SVG_DIR, f'{experience}_ethogram.svg'), format='svg', bbox_inches='tight')

## Escape Timing Histograms

In [12]:
for group_name in group_names:
    escape_hist_data = [item for sublist in ethogram_data[group_name]['escape_times'] for item in sublist]
    no_escape_hist_data = [item for sublist in ethogram_data[group_name]['no_response_times'] for item in sublist]
    plt.hist(escape_hist_data, histtype = 'step', bins = np.linspace(0, 27000, 20))
    plt.hist(no_escape_hist_data, histtype = 'step', bins = np.linspace(0, 27000, 20))
    plt.savefig(os.path.join(SVG_DIR, f'{group_name}_escape_histogram.svg'), format='svg', bbox_inches='tight')
    plt.show()

/tmp/ipykernel_16401/2361868000.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 1: Speed Trajectory Overlay

In [13]:
looming_video = '/media/arnab/My Book Duo/Data/hoylab_projects/virtualcricket_loom/naive/01-11-2024/R53-34_6.avi'
cap = cv2.VideoCapture(looming_video)
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
ret, image = cap.read()
loom_times = results['naive_adult_males_shelter']['R53-34_6']['loom_times']
centpos_xy = results['naive_adult_males_shelter']['R53-34_6']['mouse_centpos_trialwise'][0]
speed_plot = results['naive_adult_males_shelter']['R53-34_6']['mouse_speed_trialwise'][0]

In [14]:
#used video = '/media/arnab/My Book Duo/Data/hoylab_projects/virtualcricket_loom/naive/01-11-2024/R53-34_6.avi'
fig, ax = plt.subplots(1, 5, figsize = (5, 7), dpi = 300)
loom_start = loom_times[0][0]
fr = loom_start
count = 0
tr_start = loom_start-5

tr_end = tr_start + 125
for i in range(tr_start, tr_end, 25):
    print(i)
    cap.set(cv2.CAP_PROP_POS_FRAMES, i)
    ret, image = cap.read()
    # Convert to grayscale (luminance)
    if ret:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    vmax = np.nanmax(speed_plot)
    ax[count].imshow(image[:,:600], cmap = 'gray', origin='lower')  # Flip y-axis with origin='lower'
    plot_trajectory_with_speed(centpos_xy[25:-10], speed_plot[25:-10], ax[count], vmin=0, vmax=vmax, alpha = 0.4)
    print(count)
    #remove ticks and labels
    ax[count].set_xticks([])
    ax[count].set_yticks([])
    count += 1
plt.tight_layout()
plt.savefig(os.path.join(SVG_DIR, 'speed_plot_1.svg'), format='svg', bbox_inches='tight')

407
0
432
1
457
2
482
3
507
4


In [15]:
fr_list = np.array([4047, 4072, 4103, 4135, 4147])
fr_diff = np.diff(fr_list)

In [16]:
looming_video = '/media/arnab/My Book Duo/Data/hoylab_projects/virtualcricket_loom/experienced/cohort5/A64-9_6.avi'
mse_id = 'A64-9_6'

if mse_id in results.get('experienced_adult_females_shelter', {}) and len(results['experienced_adult_females_shelter'][mse_id]['mouse_centpos_trialwise']) > 0:
    cap = cv2.VideoCapture(looming_video)
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    ret, image = cap.read()

    loom_times = results['experienced_adult_females_shelter'][mse_id]['loom_times']
    centpos_xy = results['experienced_adult_females_shelter'][mse_id]['mouse_centpos_trialwise'][0]
    speed_plot = results['experienced_adult_females_shelter'][mse_id]['mouse_speed_trialwise'][0]

    fig, ax = plt.subplots(1, 5, figsize = (5, 7), dpi = 300)
    loom_start = loom_times[0][0]
    fr = loom_start
    count = 0
    tr_start = loom_start-25

    tr_end = tr_start + 125
    for i in fr_list:
        print(i)
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, image = cap.read()
        if ret:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        ax[count].imshow(image[:,:600], cmap = 'gray', origin='lower')
        plot_trajectory_with_speed(centpos_xy[5:-25], speed_plot[5:-25], ax[count], vmin=0, vmax=vmax, alpha = 0.4)
        ax[count].set_xticks([])
        ax[count].set_yticks([])
        count += 1
    plt.tight_layout()
    plt.savefig(os.path.join(SVG_DIR, 'speed_plot_2.svg'), format='svg', bbox_inches='tight')
else:
    print(f"Skipping speed_plot_2: {mse_id} has no trials after global time filtering")

Skipping speed_plot_2: A64-9_6 has no trials after global time filtering


## Colorbar

In [17]:
# Create a colorbar for the inferno colormap
fig, ax = plt.subplots(figsize=(0.3, 4), dpi = 300)
cmap = plt.cm.inferno
norm = plt.Normalize(vmin=0, vmax=vmax)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, cax=ax)
# cbar.set_label('Speed (pixels/frame)', rotation=90, labelpad=15, fontsize=10)

# Customize colorbar appearance
cbar.ax.tick_params(labelsize=20, width=1.5, length=4)
cbar.outline.set_linewidth(1.5)

plt.tight_layout()

plt.savefig(os.path.join(SVG_DIR, 'colorbar.svg'), format='svg', bbox_inches='tight')
plt.show()

/tmp/ipykernel_16401/3502799869.py:14: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
/tmp/ipykernel_16401/3502799869.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## CSV-Based Analysis Plots
Plots derived from the per-trial CSV export (`mouse_escape_data_alldata_final-22December.csv`).

In [18]:
from scipy.stats import circmean, circstd

csv_file = 'mouse_escape_data_alldata_final-22December.csv'

In [19]:
def bootstrap_median_se(data, n_bootstrap=1000):
    if len(data) == 0:
        return 0
    medians = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=len(data), replace=True)
        medians.append(np.median(sample))
    return np.std(medians)

In [20]:
def create_scatter_plot(
    csv_file, 
    variable, 
    sex=None, 
    experience=None, 
    age=None, 
    hue_variable=None,
    output_file=None, 
    show_plot=True, 
    font_sz=12, 
    vmin=None, 
    vmax=None,
    axis_fontsize=14,
    xlabel=None,
    ylabel=None,
    ytick_labels=None,
    plot_per_mouse_average=True,
    min_trials=1,
    save_svg=True,
    circular_variable=False,
):
    """
    Create a scatter plot for a specified variable from the escape analysis dataset.
    
    Fixed version that properly handles hue variables (including 'escape') when calculating per-mouse averages.
    
    Parameters:
    -----------
    csv_file : str
        Path to the CSV file
    variable : str
        Variable to plot on the y-axis (e.g., 'escape_spds_max', 'latency')
    sex : str or list, optional
        Filter by sex ('male', 'female', ['male', 'female'] for both, or 'collapse' to combine male and female data)
    experience : str, optional
        Filter by experience ('naive' or 'experienced')
    age : str, optional
        Filter by age ('adult' or 'adolescent')
    hue_variable : str, optional
        Variable name to color the dots by (e.g., 'experience', 'age', 'escape')
    output_file : str, optional
        Path to save the output figure
    show_plot : bool, default=True
        Whether to display the plot
    vmin : float, optional
        Minimum value for y-axis
    vmax : float, optional
        Maximum value for y-axis
    axis_fontsize : int, optional
        Font size for axis labels and ticks
    xlabel : str, optional
        Label for the x-axis
    ylabel : str, optional
        Label for the y-axis
    ytick_labels : list, optional
        Custom labels for the y-axis ticks
    plot_per_mouse_average : bool, default=True
        Whether to plot per-mouse averages instead of individual trials
    min_trials : int, default=1
        Minimum number of trials required to include a mouse in the analysis
    save_svg : bool, default=True
        Whether to save the figure as SVG format
    circular_variable : bool, default=False
        Whether the variable is circular (e.g., angles) and should use circular statistics
    """
    try:
        if plot_per_mouse_average:
            # Calculate per-mouse averages with proper hue variable handling
            mouse_data = calculate_mouse_averages(csv_file, variable, sex, experience, age, 
                                                 circular_variable, hue_variable)
            
            if mouse_data is None or len(mouse_data) == 0:
                print("No data available for plotting.")
                return None
            
            # Filter mice with minimum number of trials
            mouse_data = mouse_data[mouse_data[f'{variable}_count'] >= min_trials]
            print(f"Mice with at least {min_trials} trials: {len(mouse_data)}")
            
            if len(mouse_data) == 0:
                print(f"No mice have at least {min_trials} trials.")
                return None
            
            plot_data = mouse_data
            y_variable = f'{variable}_mean'
            
        else:
            # Read the CSV file for individual trial data
            df = pd.read_csv(csv_file)
            
            # Apply filters if specified
            filtered_df = df.copy()
            
            if sex is not None:
                if sex == 'collapse':
                    # Keep both male and female data but don't filter by sex
                    pass
                elif isinstance(sex, list):
                    filtered_df = filtered_df[filtered_df['sex'].isin(sex)]
                else:
                    filtered_df = filtered_df[filtered_df['sex'] == sex]
                
            if experience is not None:
                filtered_df = filtered_df[filtered_df['experience'] == experience]
                
            if age is not None:
                filtered_df = filtered_df[filtered_df['age'] == age]
            
            # Check if the variable exists in the dataset
            if variable not in filtered_df.columns:
                print(f"Error: Variable '{variable}' not found in the dataset.")
                print(f"Available variables: {', '.join(filtered_df.columns)}")
                return None
            
            # Check if we have data after filtering
            if len(filtered_df) == 0:
                print("No data matches the specified filters.")
                return None
            
            plot_data = filtered_df
            y_variable = variable
        
        # Check if hue variable exists
        if hue_variable is not None and hue_variable not in plot_data.columns:
            print(f"Error: Hue variable '{hue_variable}' not found in the dataset.")
            print(f"Available variables: {', '.join(plot_data.columns)}")
            return None
        
        # Convert radians to degrees for plotting if circular_variable is True
        if circular_variable:
            # Convert the y_variable from radians to degrees
            plot_data = plot_data.copy()  # Make a copy to avoid modifying original
            plot_data[y_variable] = np.degrees(plot_data[y_variable])
            # Take absolute values for plotting only
            plot_data[y_variable] = np.abs(plot_data[y_variable])
            print(f"\nConverted {y_variable} from radians to degrees and took absolute values for plotting only")
            
            # Filter out values greater than 150 degrees
            initial_count = len(plot_data)
            plot_data = plot_data[plot_data[y_variable] <= 140]
            filtered_count = len(plot_data)
            if initial_count > filtered_count:
                print(f"Filtered out {initial_count - filtered_count} data points with values > 150 degrees")
        
        # Print summary statistics
        print(f"\nSummary statistics for {y_variable}:")
        print(plot_data[y_variable].describe())
        
        if hue_variable:
            if sex == 'collapse':
                print(f"\nDistribution by {hue_variable} (collapsed across sex):")
                print(plot_data.groupby(hue_variable).size())
            else:
                print(f"\nDistribution by {hue_variable}:")
                print(plot_data.groupby(['sex', hue_variable]).size())
        
        # Determine the order for the 'sex' axis
        if sex == 'collapse':
            # For collapsed sex, create a single category
            sex_order = ['collapsed']
            plot_data = plot_data.copy()
            plot_data['sex'] = 'collapsed'
        elif sex is not None:
            if isinstance(sex, list):
                sex_order = []
                for s in sex:
                    if s not in sex_order:
                        sex_order.append(s)
            else:
                sex_order = [sex]
        else:
            unique_sexes = plot_data['sex'].unique().tolist()
            if 'male' in unique_sexes and 'female' in unique_sexes:
                sex_order = ['male', 'female']
            else:
                sex_order = sorted(unique_sexes)

        # Set consistent colors for male/female or use hue variable
        if hue_variable is None:
            if sex == 'collapse':
                sex_palette = {'collapsed': 'gray'}
            else:
                sex_palette = {'male': '#b4b4b44d', 'female': 'purple'}
                if len(sex_order) == 1:
                    sex_palette = {sex_order[0]: sex_palette.get(sex_order[0], '#1f77b4')}
        else:
            # Use a visually distinct palette for hue variable
            unique_hues = plot_data[hue_variable].unique()
            # Create a palette dict for boolean values if needed
            if str(plot_data[hue_variable].dtype) == 'bool':
                palette = {True: '#2ca02c', False: '#d62728'}  # Green for True, Red for False
            else:
                palette = "Set2" if len(unique_hues) <= 8 else "tab20"
        
        # Create a more informative plot
        plt.figure(figsize=(2.25, 2.5), dpi=300)
        
        if hue_variable is None:
            # Create seaborn violin plots without hue
            sns.violinplot(data=plot_data, x='sex', y=y_variable, order=sex_order, 
                          palette=sex_palette, alpha=0.3, inner=None, split=True, width=0.4)
            
            # Plot individual points with jitter
            for i, sex_val in enumerate(sex_order):
                sex_data = plot_data[plot_data['sex'] == sex_val][y_variable].dropna()
                if len(sex_data) > 0:
                    x_jitter = np.random.normal(i, 0.04, len(sex_data))
                    plt.scatter(x_jitter, sex_data, color=sex_palette[sex_val], alpha=0.8, s=20, 
                               facecolors='none', edgecolors=sex_palette[sex_val])
                    
                    # Plot mean lines with error bars
                    if circular_variable:
                        # Convert back to radians, calculate circular mean, convert to degrees, then take absolute
                        sex_data_rad = np.radians(sex_data)
                        # Remove NaN values before calculating circular mean
                        sex_data_rad_clean = sex_data_rad[~np.isnan(sex_data_rad)]
                        if len(sex_data_rad_clean) > 0:
                            mean_val_rad = circmean(sex_data_rad_clean, high=np.pi, low=-np.pi)
                            mean_val = np.abs(np.degrees(mean_val_rad))
                            # Calculate circular standard error
                            std_val_rad = circstd(sex_data_rad_clean, high=np.pi, low=-np.pi)
                            sem_val = np.degrees(std_val_rad) / np.sqrt(len(sex_data_rad_clean))
                        else:
                            mean_val = np.nan
                            sem_val = np.nan
                    else:
                        mean_val = np.mean(sex_data)
                        sem_val = np.std(sex_data) / np.sqrt(len(sex_data))
                    
                    if not np.isnan(mean_val):
                        print(f"{sex_val}: mean = {mean_val:.3f}, sem = {sem_val:.3f}")
                        plt.hlines(mean_val, i-0.2, i+0.2, color='black', linewidth=1.5, zorder=-100)
                        plt.errorbar(i, mean_val, yerr=sem_val, color='black', capsize=5, capthick=1.5, linewidth=1.5, zorder=-100)
        else:
            # Create violin plots with hue - use split=True for half violins
            sns.violinplot(data=plot_data, x='sex', y=y_variable, hue=hue_variable, 
                          order=sex_order, alpha=0.3, inner=None, split=True, width=0.4,
                          palette=palette)
            
            # Plot individual points with hue and dodge - centered on error bars
            hue_values = sorted(plot_data[hue_variable].unique())
            for i, sex_val in enumerate(sex_order):
                for j, hue_val in enumerate(hue_values):
                    subset = plot_data[(plot_data['sex'] == sex_val) & 
                                     (plot_data[hue_variable] == hue_val)][y_variable].dropna()
                    if len(subset) > 0:
                        # Calculate dodge offset to match error bar positions
                        dodge_offset = (j - len(hue_values)/2 + 0.5) * 0.2
                        x_pos = i + dodge_offset
                        
                        # Add jitter around the centered position
                        x_jitter = np.random.normal(x_pos, 0.02, len(subset))
                        
                        # Get color for this hue value
                        if isinstance(palette, dict):
                            color = palette[hue_val]
                        else:
                            # Use seaborn to get the color from the palette
                            colors = sns.color_palette(palette, len(hue_values))
                            color = colors[j]
                        
                        plt.scatter(x_jitter, subset, color=color, alpha=0.8, s=10, 
                                   facecolors=None)
            
            # Plot mean lines with error bars for each hue group
            for i, sex_val in enumerate(sex_order):
                for j, hue_val in enumerate(hue_values):
                    subset = plot_data[(plot_data['sex'] == sex_val) & 
                                     (plot_data[hue_variable] == hue_val)][y_variable].dropna()
                    if len(subset) > 0:
                        if circular_variable:
                            # Convert back to radians, calculate circular mean, convert to degrees, then take absolute
                            subset_rad = np.radians(subset)
                            # Remove NaN values before calculating circular mean
                            subset_rad_clean = subset_rad[~np.isnan(subset_rad)]
                            if len(subset_rad_clean) > 0:
                                mean_val_rad = circmean(subset_rad_clean, high=np.pi, low=-np.pi)
                                mean_val = np.abs(np.degrees(mean_val_rad))
                                # Calculate circular standard error
                                std_val_rad = circstd(subset_rad_clean, high=np.pi, low=-np.pi)
                                sem_val = np.degrees(std_val_rad) / np.sqrt(len(subset_rad_clean))
                            else:
                                mean_val = np.nan
                                sem_val = np.nan
                        else:
                            mean_val = np.mean(subset)
                            sem_val = np.std(subset) / np.sqrt(len(subset))
                        
                        # Calculate x position with dodge offset
                        dodge_offset = (j - len(hue_values)/2 + 0.5) * 0.2
                        x_pos = i + dodge_offset
                        if not np.isnan(mean_val):
                            print(f"{sex_val} {hue_variable}={hue_val}: mean = {mean_val:.3f}, sem = {sem_val:.3f}")
                            plt.hlines(mean_val, x_pos-0.1, x_pos+0.1, color='black', linewidth=1.5, zorder=-100)
                            plt.errorbar(x_pos, mean_val, yerr=sem_val, color='black', capsize=5, capthick=1.5, linewidth=1.5, zorder=-100)
            
            # Improve legend for boolean hue variables
            # if str(plot_data[hue_variable].dtype) == 'bool':
            #     handles, labels = plt.gca().get_legend_handles_labels()
            #     new_labels = ['Escaped' if str(l) == 'True' else 'No Escape' if str(l) == 'False' else str(l) for l in labels]
            #     plt.legend(handles, new_labels, title=hue_variable.capitalize())

        # Set y-axis limits if vmin or vmax are provided
        if vmin is not None or vmax is not None:
            ymin = vmin if vmin is not None else plt.ylim()[0]
            ymax = vmax if vmax is not None else plt.ylim()[1]
            plt.ylim(ymin, ymax)
        
        # Set axis labels, using custom labels if provided
        if xlabel is not None:
            plt.xlabel(xlabel, fontsize=axis_fontsize)
        else:
            if sex == 'collapse':
                plt.xlabel('Sex (Collapsed)', fontsize=axis_fontsize)
            else:
                plt.xlabel('Sex', fontsize=axis_fontsize)
        if ylabel is not None:
            plt.ylabel(ylabel, fontsize=axis_fontsize)
        else:
            if circular_variable:
                plt.ylabel(f'{y_variable} (degrees)', fontsize=axis_fontsize)
            else:
                plt.ylabel(y_variable, fontsize=axis_fontsize)
        
        # Set x-axis tick labels
        if sex == 'collapse':
            plt.xticks(range(len(sex_order)), ['All'])
        else:
            plt.xticks(range(len(sex_order)), sex_order)
        plt.tick_params(axis='both', which='major', labelsize=axis_fontsize)

        # Set custom y tick labels if provided
        if ytick_labels is not None:
            from matplotlib.ticker import FixedLocator
            # Get the current y tick locations
            yticks = plt.gca().get_yticks()
            # Only set as many labels as there are ticks
            if len(ytick_labels) == len(yticks):
                plt.gca().yaxis.set_major_locator(FixedLocator(yticks))
                plt.gca().set_yticklabels(ytick_labels, fontsize=axis_fontsize)
            else:
                print(f"Warning: Number of ytick_labels ({len(ytick_labels)}) does not match number of ticks ({len(yticks)}). Skipping custom y tick labels.")

        # Set x-axis limits
        plt.xlim(-0.5, len(sex_order) - 0.5)
        
        # Remove right and top spines
        sns.despine()
        
        # Adjust layout
        plt.tight_layout()
        
        # Save the figure if requested
        if output_file and save_svg:
            # Always save as SVG, regardless of extension
            base, ext = os.path.splitext(output_file)
            svg_file = base + ".svg"
            plt.savefig(svg_file, format='svg', dpi=300, bbox_inches='tight')
            print(f"Figure saved to {svg_file}")
        elif output_file and not save_svg:
            plt.savefig(output_file, dpi=300, bbox_inches='tight')
            print(f"Figure saved to {output_file}")
        
        # Show the plot if requested
        if show_plot:
            plt.show()
            
        return plot_data  # Return the plot data for further analysis
            
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        return None


def calculate_mouse_averages(csv_file, variable, sex=None, experience=None, age=None, 
                            circular_variable=False, hue_variable=None):
    """
    Calculate average values per mouse for a specified variable, with support for circular statistics.
    
    Fixed version that properly includes hue_variable in the grouping columns.
    
    Parameters:
    -----------
    csv_file : str
        Path to the CSV file
    variable : str
        Variable to calculate averages for (e.g., 'escape_spds_max', 'latency')
    sex : str or list, optional
        Filter by sex ('male', 'female', ['male', 'female'] for both, or 'collapse' to combine male and female data)
    experience : str, optional
        Filter by experience ('naive' or 'experienced')
    age : str, optional
        Filter by age ('adult' or 'adolescent')
    circular_variable : bool, default=False
        Whether the variable is circular (e.g., angles) and should use circular statistics
    hue_variable : str, optional
        Additional grouping variable (e.g., 'escape') to include in the aggregation
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with average values for each mouse
    """
    try:
        # Read the CSV file
        df = pd.read_csv(csv_file)
        
        # Print basic information about the dataset
        print(f"Dataset loaded: {len(df)} rows, {len(df.columns)} columns")
        print(f"Total mice: {df['mouse_id'].nunique()}")
        
        # Apply filters if specified
        filtered_df = df.copy()
        filters_applied = []
        
        if sex is not None:
            if sex == 'collapse':
                # Keep both male and female data but modify sex column for grouping
                filtered_df['sex'] = 'collapsed'
                filters_applied.append("sex='collapse' (combining male and female)")
            elif isinstance(sex, list):
                filtered_df = filtered_df[filtered_df['sex'].isin(sex)]
                filters_applied.append(f"sex in {sex}")
            else:
                filtered_df = filtered_df[filtered_df['sex'] == sex]
                filters_applied.append(f"sex='{sex}'")
            
        if experience is not None:
            filtered_df = filtered_df[filtered_df['experience'] == experience]
            filters_applied.append(f"experience='{experience}'")
            
        if age is not None:
            filtered_df = filtered_df[filtered_df['age'] == age]
            filters_applied.append(f"age='{age}'")
        
        if filters_applied:
            print(f"Filters applied: {', '.join(filters_applied)}")
        
        # Check if the variable exists in the dataset
        if variable not in filtered_df.columns:
            print(f"Error: Variable '{variable}' not found in the dataset.")
            print(f"Available variables: {', '.join(filtered_df.columns)}")
            return None
        
        # Check if we have data after filtering
        if len(filtered_df) == 0:
            print("No data matches the specified filters.")
            return None
        
        # Get all grouping columns that exist in the dataframe
        grouping_cols = ['mouse_id', 'sex']
        if 'age' in filtered_df.columns:
            grouping_cols.append('age')
        if 'experience' in filtered_df.columns:
            grouping_cols.append('experience')
        
        # IMPORTANT FIX: Include hue_variable in grouping if specified
        if hue_variable and hue_variable in filtered_df.columns:
            if hue_variable not in grouping_cols:
                grouping_cols.append(hue_variable)
        
        # Calculate averages per mouse (only for non-null values)
        if circular_variable:
            # Use circular mean for angular data (in radians)
            def circular_mean_func(x):
                # Remove NaN values before calculating circular mean
                x_clean = x.dropna()
                if len(x_clean) == 0:
                    return np.nan
                return circmean(x_clean, high=np.pi, low=-np.pi)
            
            mouse_averages = filtered_df.groupby(grouping_cols).agg({
                variable: ['count', circular_mean_func]
            }).reset_index()
        else:
            # Use regular mean for non-circular data
            mouse_averages = filtered_df.groupby(grouping_cols).agg({
                variable: ['count', 'mean']
            }).reset_index()
        
        # Flatten column names
        new_columns = grouping_cols + [f'{variable}_count', f'{variable}_mean']
        mouse_averages.columns = new_columns
        
        # Remove mice with no valid data for the variable
        mouse_averages = mouse_averages.dropna(subset=[f'{variable}_mean'])
        
        # For circular variables, filter out average values greater than 150 degrees
        if circular_variable:
            # Convert to degrees for filtering
            mouse_averages_degrees = mouse_averages.copy()
            mouse_averages_degrees[f'{variable}_mean'] = np.abs(np.degrees(mouse_averages_degrees[f'{variable}_mean']))
            
            initial_count = len(mouse_averages)
            mouse_averages = mouse_averages[mouse_averages_degrees[f'{variable}_mean'] <= 150]
            filtered_count = len(mouse_averages)
            
            if initial_count > filtered_count:
                print(f"Filtered out {initial_count - filtered_count} mice with average values > 150 degrees")
        
        print(f"\nAfter filtering: {len(mouse_averages)} mice/conditions with valid {variable} data")
        print(f"Values per mouse: {mouse_averages[f'{variable}_count'].describe()}")
        print(f"Average {variable}: {mouse_averages[f'{variable}_mean'].describe()}")
        
        return mouse_averages
        
    except Exception as e:
        print(f"Error calculating mouse averages: {e}")
        import traceback
        traceback.print_exc()
        return None


In [21]:
def calculate_escape_probabilities(csv_file, sex=None, experience=None, age=None, escape_key='escape'):
    """
    Calculate escape probabilities for each mouse from the escape analysis dataset.
    
    Parameters:
    -----------
    csv_file : str
        Path to the CSV file
    sex : str or list, optional
        Filter by sex ('male', 'female', or ['male', 'female'] for both)
    experience : str, optional
        Filter by experience ('naive' or 'experienced')
    age : str, optional
        Filter by age ('adult' or 'adolescent')
    escape_key : str, default='escape'
        Column name to use for escape data
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with escape probabilities for each mouse
    """
    try:
        # Read the CSV file
        df = pd.read_csv(csv_file)
        
        # Check if escape_key exists in the dataframe
        if escape_key not in df.columns:
            print(f"Error: Column '{escape_key}' not found in the dataset.")
            print(f"Available columns: {', '.join(df.columns)}")
            return None
        
        # Convert escape column to boolean if it's string
        if df[escape_key].dtype == 'object':
            df[escape_key] = df[escape_key].map({'True': True, 'False': False})
        
        # Print basic information about the dataset
        print(f"Dataset loaded: {len(df)} rows, {len(df.columns)} columns")
        print(f"Total mice: {df['mouse_id'].nunique()}")
        
        # Apply filters if specifiednaive
        filtered_df = df.copy()
        filters_applied = []
        
        if sex is not None:
            if isinstance(sex, list):
                filtered_df = filtered_df[filtered_df['sex'].isin(sex)]
                filters_applied.append(f"sex in {sex}")
            else:
                filtered_df = filtered_df[filtered_df['sex'] == sex]
                filters_applied.append(f"sex='{sex}'")
            
        if experience is not None:
            filtered_df = filtered_df[filtered_df['experience'] == experience]
            filters_applied.append(f"experience='{experience}'")
            
        if age is not None:
            filtered_df = filtered_df[filtered_df['age'] == age]
            filters_applied.append(f"age='{age}'")
        
        if filters_applied:
            print(f"Filters applied: {', '.join(filters_applied)}")
        
        # Check if we have data after filtering
        if len(filtered_df) == 0:
            print("No data matches the specified filters.")
            return None
        
        # Calculate escape probabilities for each mouse
        escape_probs = filtered_df.groupby(['mouse_id', 'sex', 'age', 'experience']).agg({
            escape_key: ['count', 'sum', 'mean']
        }).reset_index()
        
        # Flatten column names
        escape_probs.columns = ['mouse_id', 'sex', 'age', 'experience', 'total_trials', 'total_escapes', 'escape_probability']
        
        print(f"\nAfter filtering: {len(escape_probs)} mice")
        print(f"Trials per mouse: {escape_probs['total_trials'].describe()}")
        print(f"Escape probabilities: {escape_probs['escape_probability'].describe()}")
        
        return escape_probs
        
    except Exception as e:
        print(f"Error calculating escape probabilities: {e}")
        import traceback
        traceback.print_exc()
        return None

def create_escape_probability_plot(csv_file, sex=None, experience=None, age=None, escape_key='escape', 
                                 hue_variable=None, output_file=None, show_plot=True, 
                                 font_size=12, min_trials=3):
    """
    Create a scatter plot showing escape probabilities for each mouse.
    
    Parameters:
    -----------
    csv_file : str
        Path to the CSV file
    sex : str or list, optional
        Filter by sex ('male', 'female', or ['male', 'female'] for both)
    experience : str, optional
        Filter by experience ('naive' or 'experienced')
    age : str, optionalnaive
        Filter by age ('adult' or 'adolescent')
    hue_variable : str, optional
        Variable name to color the dots by (e.g., 'experience', 'age')
    output_file : str, optional
        Path to save the output figure
    show_plot : bool, default=True
        Whether to display the plot
    font_size : int, default=12
        Base font size for the plot
    min_trials : int, default=3
        Minimum number of trials required to include a mouse in the analysis
    """
    try:
        # Calculate escape probabilities
        escape_data = calculate_escape_probabilities(csv_file, sex, experience, age, escape_key)
        
        if escape_data is None or len(escape_data) == 0:
            print("No data available for plotting.")
            return None
        
        # Filter mice with minimum number of trials
        escape_data = escape_data[escape_data['total_trials'] >= min_trials]
        print(f"Mice with at least {min_trials} trials: {len(escape_data)}")
        
        if len(escape_data) == 0:
            print(f"No mice have at least {min_trials} trials.")
            return None
        
        # Check if hue variable exists
        if hue_variable is not None and hue_variable not in escape_data.columns:
            print(f"Error: Hue variable '{hue_variable}' not found in the dataset.")
            print(f"Available variables: {', '.join(escape_data.columns)}")
            return None
        
        # Print summary statistics
        print(f"\nSummary statistics for escape_probability:")
        print(escape_data['escape_probability'].describe())
        
        # Determine the order for the 'sex' axis
        if sex is not None:
            if isinstance(sex, list):
                sex_order = []
                for s in sex:
                    if s not in sex_order:
                        sex_order.append(s)
            else:
                sex_order = [sex]
        else:
            unique_sexes = escape_data['sex'].unique().tolist()
            if 'male' in unique_sexes and 'female' in unique_sexes:
                sex_order = ['male', 'female']
            else:
                sex_order = sorted(unique_sexes)

        # Set consistent colors for male/female
        sex_palette = {'male': '#b4b4b44d', 'female': 'purple'}
        if len(sex_order) == 1:
            sex_palette = {sex_order[0]: sex_palette.get(sex_order[0], '#1f77b4')}
        
        # Create a more informative plot
        plt.figure(figsize=(2.25, 2.5), dpi=300)
        
        # Create seaborn violin plots
        sns.violinplot(data=escape_data, x='sex', y='escape_probability', order=sex_order, 
                      palette=sex_palette, alpha=0.3, inner=None, split=True, width=0.4)
        
        # Plot individual points with jitter and calculate median with error bars
        median_values = []
        error_values = []
        x_positions = []
        
        for i, sex_val in enumerate(sex_order):
            sex_data = escape_data[escape_data['sex'] == sex_val]['escape_probability'].dropna()
            if len(sex_data) > 0:
                x_jitter = np.random.normal(i, 0.04, len(sex_data))
                plt.scatter(x_jitter, sex_data, color=sex_palette[sex_val], alpha=0.8, s=20, 
                           facecolors='none', edgecolors=sex_palette[sex_val])
                
                # Calculate median and error
                median_val = np.median(sex_data)
                error_val = bootstrap_median_se(sex_data)
                
                # Ensure error bars stay within [0,1] bounds
                lower_error = min(error_val, median_val)  # Don't go below 0
                upper_error = min(error_val, 1 - median_val)  # Don't go above 1
                
                median_values.append(median_val)
                error_values.append([lower_error, upper_error])
                x_positions.append(i)
        
        # Add error bars for medians with asymmetric bounds
        if error_values:
            error_array = np.array(error_values).T  # Transpose to get [lower_errors, upper_errors]
            plt.errorbar(x_positions, median_values, yerr=error_array, 
                        fmt='none', color='black', capsize=3, capthick=1.5, linewidth=1.5, zorder=-100)
        
        # Plot median lines
        for i, (x_pos, median_val) in enumerate(zip(x_positions, median_values)):
            plt.hlines(median_val, x_pos-0.2, x_pos+0.2, color='black', linewidth=1.5, zorder=-100)

        # Set y-axis limits for probability
        plt.ylim(-0.05, 1.05)
        
        # Set axis labels
        plt.ylabel(f'{escape_key.capitalize()} Probability', fontsize=font_size + 2)
        
        plt.xticks(range(len(sex_order)), sex_order)
        plt.tick_params(axis='both', which='major', labelsize=font_size + 2)

        # Set x-axis limits
        plt.xlim(-0.5, len(sex_order) - 0.5)
        
        # Remove right and top spines
        sns.despine()
        
        # Adjust layout
        plt.tight_layout()
        
        # Save the figure if requested
        if output_file:
            # Always save as SVG, regardless of extension
            base, ext = os.path.splitext(output_file)
            svg_file = base + ".svg"
            plt.savefig(svg_file, format='svg', dpi=300, bbox_inches='tight')
            print(f"Figure saved to {svg_file}")
        
        # Show the plot if requested
        if show_plot:
            plt.show()
        
        return escape_data
        
    except Exception as e:
        print(f"Error creating plot: {e}")
        import traceback
        traceback.print_exc()
        return None


### Escape Speed

In [22]:
variable = 'escape_spds_max'
for experience in ['naive', 'experienced']:
    for age in ['adolescent', 'adult']:
        create_scatter_plot(csv_file,
            variable, 
            sex = ['male', 'female'], 
            experience=experience, 
            age=age,
            show_plot=True,
            vmin=0,
            vmax=110,
            axis_fontsize=8, 
            xlabel=' ',
            ylabel=' ',
            output_file=os.path.join(SVG_DIR, f'{variable}_{age}_{experience}_plot.svg'),
            save_svg=True
        )

Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='naive', age='adolescent'

After filtering: 7 mice/conditions with valid escape_spds_max data
Values per mouse: count    7.000000
mean     1.285714
std      0.487950
min      1.000000
25%      1.000000
50%      1.000000
75%      1.500000
max      2.000000
Name: escape_spds_max_count, dtype: float64
Average escape_spds_max: count     7.000000
mean     60.271583
std      21.559339
min      32.800952
25%      46.821206
50%      57.772989
75%      73.092579
max      91.499567
Name: escape_spds_max_mean, dtype: float64
Mice with at least 1 trials: 7

Summary statistics for escape_spds_max_mean:
count     7.000000
mean     60.271583
std      21.559339
min      32.800952
25%      46.821206
50%      57.772989
75%      73.092579
max      91.499567
Name: escape_spds_max_mean, dtype: float64


/tmp/ipykernel_16401/2709766471.py:200: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_data, x='sex', y=y_variable, order=sex_order,


male: mean = 65.774, sem = 8.678
female: mean = 46.516, sem = 9.698
Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/escape_spds_max_adolescent_naive_plot.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='naive', age='adult'

After filtering: 6 mice/conditions with valid escape_spds_max data
Values per mouse: count    6.000000
mean     1.166667
std      0.408248
min      1.000000
25%      1.000000
50%      1.000000
75%      1.000000
max      2.000000
Name: escape_spds_max_count, dtype: float64
Average escape_spds_max: count     6.000000
mean     56.988337
std      21.689848
min      38.565070
25%      46.805323
50%      48.769319
75%      57.825801
max      98.831029
Name: escape_spds_max_mean, dtype: float64
Mice with at least 1 trials: 6

Summary statistics for escape_spds_max_mean:
count     6.000000
mean     56.988337
std      21.689848
min      38.565070
25%      46.805323
50

/tmp/ipykernel_16401/2709766471.py:366: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16401/2709766471.py:200: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_data, x='sex', y=y_variable, order=sex_order,


female: mean = 52.137, sem = 3.540


...

Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/escape_spds_max_adult_naive_plot.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='experienced', age='adolescent'

After filtering: 6 mice/conditions with valid escape_spds_max data
Values per mouse: count    6.000000
mean     1.166667
std      0.408248
min      1.000000
25%      1.000000
50%      1.000000
75%      1.000000
max      2.000000
Name: escape_spds_max_count, dtype: float64
Average escape_spds_max: count     6.000000
mean     46.052804
std      13.136689
min      36.226608
25%      36.762169
50%      39.568923
75%      53.035083
max      67.601981
Name: escape_spds_max_mean, dtype: float64
Mice with at least 1 trials: 6

Summary statistics for escape_spds_max_mean:
count     6.000000
mean     46.052804
std      13.136689
min      36.226608
25%      36.762169
50%      39.568923
75%      53.035083
max      67.601981
Name: e

Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='experienced', age='adult'

After filtering: 5 mice/conditions with valid escape_spds_max data
Values per mouse: count    5.00000
mean     1.80000
std      1.30384
min      1.00000
25%      1.00000
50%      1.00000
75%      2.00000
max      4.00000
Name: escape_spds_max_count, dtype: float64
Average escape_spds_max: count     5.000000
mean     51.968761
std      16.961103
min      41.777996
25%      44.137869
50%      45.417678
75%      46.356599
max      82.153666
Name: escape_spds_max_mean, dtype: float64
Mice with at least 1 trials: 5

Summary statistics for escape_spds_max_mean:
count     5.000000
mean     51.968761
std      16.961103
min      41.777996
25%      44.137869
50%      45.417678
75%      46.356599
max      82.153666
Name: escape_spds_max_mean, dtype: float64
male: mean = 45.304, sem = 0.525
female: mean = 61.966, sem = 14.275


/tmp/ipykernel_16401/2709766471.py:366: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16401/2709766471.py:196: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(2.25, 2.5), dpi=300)
/tmp/ipykernel_16401/2709766471.py:200: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_data, x='sex', y=y_variable, order=sex_order,


Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/escape_spds_max_adult_experienced_plot.svg


/tmp/ipykernel_16401/2709766471.py:366: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Freeze Duration

In [23]:
variable = 'freeze_duration'
for experience in ['naive', 'experienced']:
    for age in ['adolescent', 'adult']:
        create_scatter_plot(csv_file,
            variable, 
            sex=['male', 'female'], 
            experience=experience, 
            age=age,
            show_plot=True,
            vmin=0,
            vmax=2,
            axis_fontsize=8, 
            xlabel=' ',
            ylabel=' ',
            output_file=os.path.join(SVG_DIR, f'{variable}_{age}_{experience}_plot.svg'),
            save_svg=True
        )

/tmp/ipykernel_16401/2709766471.py:200: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_data, x='sex', y=y_variable, order=sex_order,


Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='naive', age='adolescent'

After filtering: 7 mice/conditions with valid freeze_duration data
Values per mouse: count    7.000000
mean     1.142857
std      0.377964
min      1.000000
25%      1.000000
50%      1.000000
75%      1.000000
max      2.000000
Name: freeze_duration_count, dtype: float64
Average freeze_duration: count    7.000000
mean     1.261905
std      0.908077
min      0.666667
25%      0.783333
50%      1.000000
75%      1.166667
max      3.266667
Name: freeze_duration_mean, dtype: float64
Mice with at least 1 trials: 7

Summary statistics for freeze_duration_mean:
count    7.000000
mean     1.261905
std      0.908077
min      0.666667
25%      0.783333
50%      1.000000
75%      1.166667
max      3.266667
Name: freeze_duration_mean, dtype: float64


male: mean = 0.956, sem = 0.127
female: mean = 1.492, sem = 0.519
Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/freeze_duration_adolescent_naive_plot.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='naive', age='adult'

After filtering: 16 mice/conditions with valid freeze_duration data
Values per mouse: count    16.000000
mean      1.375000
std       0.619139
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       3.000000
Name: freeze_duration_count, dtype: float64
Average freeze_duration: count    16.000000
mean      1.353472
std       0.802578
min       0.500000
25%       0.837500
50%       1.050000
75%       1.720833
max       3.200000
Name: freeze_duration_mean, dtype: float64
Mice with at least 1 trials: 16

Summary statistics for freeze_duration_mean:
count    16.000000
mean      1.353472
std       0.802578
min       0.500000
25%       0.8

/tmp/ipykernel_16401/2709766471.py:366: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16401/2709766471.py:200: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_data, x='sex', y=y_variable, order=sex_order,


male: mean = 1.326, sem = 0.256
female: mean = 1.381, sem = 0.292


...

Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/freeze_duration_adult_naive_plot.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='experienced', age='adolescent'

After filtering: 10 mice/conditions with valid freeze_duration data
Values per mouse: count    10.000000
mean      1.800000
std       1.135292
min       1.000000
25%       1.000000
50%       1.000000
75%       2.750000
max       4.000000
Name: freeze_duration_count, dtype: float64
Average freeze_duration: count    10.000000
mean      0.711111
std       0.129259
min       0.533333
25%       0.633333
50%       0.666667
75%       0.783333
max       0.977778
Name: freeze_duration_mean, dtype: float64
Mice with at least 1 trials: 10

Summary statistics for freeze_duration_mean:
count    10.000000
mean      0.711111
std       0.129259
min       0.533333
25%       0.633333
50%       0.666667
75%       0.783333
max       0.9777

Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/freeze_duration_adolescent_experienced_plot.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='experienced', age='adult'

After filtering: 12 mice/conditions with valid freeze_duration data
Values per mouse: count    12.000000
mean      1.666667
std       0.651339
min       1.000000
25%       1.000000
50%       2.000000
75%       2.000000
max       3.000000
Name: freeze_duration_count, dtype: float64
Average freeze_duration: count    12.000000
mean      0.937037
std       0.264886
min       0.533333
25%       0.720833
50%       0.950000
75%       1.179167
max       1.316667
Name: freeze_duration_mean, dtype: float64
Mice with at least 1 trials: 12

Summary statistics for freeze_duration_mean:
count    12.000000
mean      0.937037
std       0.264886
min       0.533333
25%       0.720833
50%       0.950000
75%       1.179167
max       

/tmp/ipykernel_16401/2709766471.py:366: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16401/2709766471.py:200: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_data, x='sex', y=y_variable, order=sex_order,


male: mean = 0.828, sem = 0.076
female: mean = 1.090, sem = 0.107
Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/freeze_duration_adult_experienced_plot.svg


/tmp/ipykernel_16401/2709766471.py:366: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Escape Latency

In [24]:
variable = 'latency'
for experience in ['naive', 'experienced']:
    for age in ['adolescent', 'adult']:
        create_scatter_plot(csv_file,
            variable, 
            sex=['male', 'female'], 
            experience=experience, 
            age=age,
            show_plot=True,
            vmin=0,
            vmax=70,
            axis_fontsize=8, 
            xlabel=' ',
            ylabel=' ',
            ytick_labels= [f"{x/30:.1f}" for x in [0, 10, 20, 30, 40, 50, 60, 70]],
            output_file=os.path.join(SVG_DIR, f'{variable}_{age}_{experience}_plot.svg'),
            save_svg=True)

Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='naive', age='adolescent'

After filtering: 8 mice/conditions with valid latency data
Values per mouse: count    8.00000
mean     1.25000
std      0.46291
min      1.00000
25%      1.00000
50%      1.00000
75%      1.25000
max      2.00000
Name: latency_count, dtype: float64
Average latency: count     8.000000
mean     34.937500
std      31.055753
min       0.000000
25%       8.875000
50%      31.500000
75%      54.000000
max      84.000000
Name: latency_mean, dtype: float64
Mice with at least 1 trials: 8

Summary statistics for latency_mean:
count     8.000000
mean     34.937500
std      31.055753
min       0.000000
25%       8.875000
50%      31.500000
75%      54.000000
max      84.000000
Name: latency_mean, dtype: float64


/tmp/ipykernel_16401/2709766471.py:200: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_data, x='sex', y=y_variable, order=sex_order,


male: mean = 27.750, sem = 12.169
female: mean = 56.500, sem = 6.718


/tmp/ipykernel_16401/2709766471.py:366: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16401/2709766471.py:200: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_data, x='sex', y=y_variable, order=sex_order,


Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/latency_adolescent_naive_plot.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='naive', age='adult'

After filtering: 6 mice/conditions with valid latency data
Values per mouse: count    6.000000
mean     1.166667
std      0.408248
min      1.000000
25%      1.000000
50%      1.000000
75%      1.000000
max      2.000000
Name: latency_count, dtype: float64
Average latency: count     6.000000
mean     34.333333
std      28.465183
min       8.000000
25%       8.750000
50%      32.000000
75%      54.500000
max      71.000000
Name: latency_mean, dtype: float64
Mice with at least 1 trials: 6

Summary statistics for latency_mean:
count     6.000000
mean     34.333333
std      28.465183
min       8.000000
25%       8.750000
50%      32.000000
75%      54.500000
max      71.000000
Name: latency_mean, dtype: float64


...

/tmp/ipykernel_16401/2709766471.py:366: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16401/2709766471.py:200: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_data, x='sex', y=y_variable, order=sex_order,


Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/latency_adolescent_experienced_plot.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='experienced', age='adult'

After filtering: 5 mice/conditions with valid latency data
Values per mouse: count    5.00000
mean     1.80000
std      1.30384
min      1.00000
25%      1.00000
50%      1.00000
75%      2.00000
max      4.00000
Name: latency_count, dtype: float64
Average latency: count     5.000000
mean     45.100000
std      24.480605
min      22.000000
25%      24.000000
50%      38.500000
75%      64.000000
max      77.000000
Name: latency_mean, dtype: float64
Mice with at least 1 trials: 5

Summary statistics for latency_mean:
count     5.000000
mean     45.100000
std      24.480605
min      22.000000
25%      24.000000
50%      38.500000
75%      64.000000
max      77.000000
Name: latency_mean, dtype: float64
male: mean = 55.000, s

/tmp/ipykernel_16401/2709766471.py:366: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16401/2709766471.py:200: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_data, x='sex', y=y_variable, order=sex_order,


female: mean = 30.250, sem = 5.834
Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/latency_adult_experienced_plot.svg


/tmp/ipykernel_16401/2709766471.py:366: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Freeze Probability

In [25]:
variable = 'freeze'
for experience in ['naive', 'experienced']:
    for age in ['adolescent', 'adult']:
        result = create_escape_probability_plot(csv_file,
            sex=['male', 'female'],
            experience=experience,
            escape_key=variable,
            age=age,
            output_file=os.path.join(SVG_DIR, f'freeze_probability_{experience}_{age}_plot.svg'),
            show_plot=True,
            font_size=8,
            min_trials=0    
        )

Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='naive', age='adolescent'

After filtering: 16 mice
Trials per mouse: count    16.000000
mean      1.625000
std       0.806226
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       3.000000
Name: total_trials, dtype: float64
Escape probabilities: count    16.000000
mean      0.343750
std       0.440827
min       0.000000
25%       0.000000
50%       0.000000
75%       0.750000
max       1.000000
Name: escape_probability, dtype: float64
Mice with at least 0 trials: 16

Summary statistics for escape_probability:
count    16.000000
mean      0.343750
std       0.440827
min       0.000000
25%       0.000000
50%       0.000000
75%       0.750000
max       1.000000
Name: escape_probability, dtype: float64


/tmp/ipykernel_16401/1568402581.py:167: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=escape_data, x='sex', y='escape_probability', order=sex_order,


Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/freeze_probability_naive_adolescent_plot.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='naive', age='adult'

After filtering: 19 mice
Trials per mouse: count    19.000000
mean      1.631579
std       0.760886
min       1.000000
25%       1.000000
50%       2.000000
75%       2.000000
max       4.000000
Name: total_trials, dtype: float64
Escape probabilities: count    19.000000
mean      0.750000
std       0.381881
min       0.000000
25%       0.500000
50%       1.000000
75%       1.000000
max       1.000000
Name: escape_probability, dtype: float64
Mice with at least 0 trials: 19

Summary statistics for escape_probability:
count    19.000000
mean      0.750000
std       0.381881
min       0.000000
25%       0.500000
50%       1.000000
75%       1.000000
max       1.000000
Name: escape_probability, dtype: float64


/tmp/ipykernel_16401/1568402581.py:232: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16401/1568402581.py:167: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=escape_data, x='sex', y='escape_probability', order=sex_order,


Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/freeze_probability_naive_adult_plot.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='experienced', age='adolescent'

After filtering: 16 mice
Trials per mouse: count    16.00000
mean      2.56250
std       1.41274
min       1.00000
25%       1.75000
50%       2.00000
75%       4.00000
max       5.00000
Name: total_trials, dtype: float64
Escape probabilities: count    16.000000
mean      0.408333
std       0.411366
min       0.000000
25%       0.000000
50%       0.291667
75%       0.700000
max       1.000000
Name: escape_probability, dtype: float64
Mice with at least 0 trials: 16

Summary statistics for escape_probability:
count    16.000000
mean      0.408333
std       0.411366
min       0.000000
25%       0.000000
50%       0.291667
75%       0.700000
max       1.000000
Name: escape_probability, dtype: float64


/tmp/ipykernel_16401/1568402581.py:232: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16401/1568402581.py:167: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=escape_data, x='sex', y='escape_probability', order=sex_order,


Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/freeze_probability_experienced_adolescent_plot.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='experienced', age='adult'

After filtering: 16 mice
Trials per mouse: count    16.000000
mean      2.437500
std       1.152895
min       1.000000
25%       1.750000
50%       2.000000
75%       3.250000
max       4.000000
Name: total_trials, dtype: float64
Escape probabilities: count    16.000000
mean      0.593750
std       0.421500
min       0.000000
25%       0.250000
50%       0.583333
75%       1.000000
max       1.000000
Name: escape_probability, dtype: float64
Mice with at least 0 trials: 16

Summary statistics for escape_probability:
count    16.000000
mean      0.593750
std       0.421500
min       0.000000
25%       0.250000
50%       0.583333
75%       1.000000
max       1.000000
Name: escape_probability, dtype: float64


/tmp/ipykernel_16401/1568402581.py:232: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16401/1568402581.py:167: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=escape_data, x='sex', y='escape_probability', order=sex_order,


Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/freeze_probability_experienced_adult_plot.svg


/tmp/ipykernel_16401/1568402581.py:232: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Escape Probability (Per-Mouse)

In [26]:
variable = 'escape'
for experience in ['naive', 'experienced']:
    for age in ['adolescent', 'adult']:
        result = create_escape_probability_plot(
            csv_file,
            sex=['male', 'female'],
            experience=experience,
            age=age,
            escape_key='escape',
            show_plot=True,
            font_size=8,
            min_trials=0,
            output_file=os.path.join(SVG_DIR, f'{variable}_probability_{age}_{experience}_plot.svg'),
        )

Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='naive', age='adolescent'

After filtering: 16 mice
Trials per mouse: count    16.000000
mean      1.625000
std       0.806226
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       3.000000
Name: total_trials, dtype: float64
Escape probabilities: count    16.000000
mean      0.416667
std       0.459468
min       0.000000
25%       0.000000
50%       0.250000
75%       1.000000
max       1.000000
Name: escape_probability, dtype: float64
Mice with at least 0 trials: 16

Summary statistics for escape_probability:
count    16.000000
mean      0.416667
std       0.459468
min       0.000000
25%       0.000000
50%       0.250000
75%       1.000000
max       1.000000
Name: escape_probability, dtype: float64


/tmp/ipykernel_16401/1568402581.py:167: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=escape_data, x='sex', y='escape_probability', order=sex_order,


Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/escape_probability_adolescent_naive_plot.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='naive', age='adult'

After filtering: 19 mice
Trials per mouse: count    19.000000
mean      1.631579
std       0.760886
min       1.000000
25%       1.000000
50%       2.000000
75%       2.000000
max       4.000000
Name: total_trials, dtype: float64
Escape probabilities: count    19.000000
mean      0.210526
std       0.346241
min       0.000000
25%       0.000000
50%       0.000000
75%       0.500000
max       1.000000
Name: escape_probability, dtype: float64
Mice with at least 0 trials: 19

Summary statistics for escape_probability:
count    19.000000
mean      0.210526
std       0.346241
min       0.000000
25%       0.000000
50%       0.000000
75%       0.500000
max       1.000000
Name: escape_probability, dtype: float64


/tmp/ipykernel_16401/1568402581.py:232: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16401/1568402581.py:167: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=escape_data, x='sex', y='escape_probability', order=sex_order,


Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/escape_probability_adult_naive_plot.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='experienced', age='adolescent'

After filtering: 16 mice
Trials per mouse: count    16.00000
mean      2.56250
std       1.41274
min       1.00000
25%       1.75000
50%       2.00000
75%       4.00000
max       5.00000
Name: total_trials, dtype: float64
Escape probabilities: count    16.000000
mean      0.173958
std       0.286290
min       0.000000
25%       0.000000
50%       0.000000
75%       0.270833
max       1.000000
Name: escape_probability, dtype: float64
Mice with at least 0 trials: 16

Summary statistics for escape_probability:
count    16.000000
mean      0.173958
std       0.286290
min       0.000000
25%       0.000000
50%       0.000000
75%       0.270833
max       1.000000
Name: escape_probability, dtype: float64


/tmp/ipykernel_16401/1568402581.py:232: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16401/1568402581.py:167: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=escape_data, x='sex', y='escape_probability', order=sex_order,


Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/escape_probability_adolescent_experienced_plot.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='experienced', age='adult'

After filtering: 16 mice
Trials per mouse: count    16.000000
mean      2.437500
std       1.152895
min       1.000000
25%       1.750000
50%       2.000000
75%       3.250000
max       4.000000
Name: total_trials, dtype: float64
Escape probabilities: count    16.000000
mean      0.177083
std       0.340037
min       0.000000
25%       0.000000
50%       0.000000
75%       0.250000
max       1.000000
Name: escape_probability, dtype: float64
Mice with at least 0 trials: 16

Summary statistics for escape_probability:
count    16.000000
mean      0.177083
std       0.340037
min       0.000000
25%       0.000000
50%       0.000000
75%       0.250000
max       1.000000
Name: escape_probability, dtype: float64


/tmp/ipykernel_16401/1568402581.py:232: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16401/1568402581.py:167: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=escape_data, x='sex', y='escape_probability', order=sex_order,


Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/escape_probability_adult_experienced_plot.svg


/tmp/ipykernel_16401/1568402581.py:232: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Cricket Azimuth

In [27]:
variable = 'cricket_az'
for experience in ['naive', 'experienced']:
    for age in ['adolescent', 'adult']:
        create_scatter_plot(csv_file, 
            variable, 
            sex= 'collapse', 
            experience=experience, 
            age=age, 
            hue_variable='escape',
            show_plot=True, 
            font_sz=8, 
            axis_fontsize=8,
            plot_per_mouse_average=True,
            vmin = 0,
            vmax = 150,
            min_trials=1,
            circular_variable=True,
            output_file=os.path.join(SVG_DIR, f'{variable}_{age}_{experience}_plot_combined.svg'),
            save_svg=True)

Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex='collapse' (combining male and female), experience='naive', age='adolescent'
Filtered out 2 mice with average values > 150 degrees

After filtering: 17 mice/conditions with valid cricket_az data
Values per mouse: count    17.000000
mean      1.411765
std       0.712287
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       3.000000
Name: cricket_az_count, dtype: float64
Average cricket_az: count    17.000000
mean      0.011917
std       1.037190
min      -1.914417
25%      -0.685258
50%       0.003261
75%       0.986618
max       1.586625
Name: cricket_az_mean, dtype: float64
Mice with at least 1 trials: 17

Converted cricket_az_mean from radians to degrees and took absolute values for plotting only

Summary statistics for cricket_az_mean:
count     17.000000
mean      46.344909
std       35.354285
min        0.186819
25%       14.249542
50%       53.926320
75%       73.371720
max   

Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/cricket_az_adolescent_naive_plot_combined.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex='collapse' (combining male and female), experience='naive', age='adult'
Filtered out 1 mice with average values > 150 degrees

After filtering: 22 mice/conditions with valid cricket_az data
Values per mouse: count    22.000000
mean      1.363636
std       0.726731
min       1.000000
25%       1.000000
50%       1.000000
75%       1.750000
max       4.000000
Name: cricket_az_count, dtype: float64
Average cricket_az: count    22.000000
mean     -0.237460
std       1.082835
min      -2.002276
25%      -0.659814
50%      -0.338652
75%       0.375071
max       2.289274
Name: cricket_az_mean, dtype: float64
Mice with at least 1 trials: 22

Converted cricket_az_mean from radians to degrees and took absolute values for plotting only

Summary statistics for cricket_az_mean:
count     2

/tmp/ipykernel_16401/2709766471.py:366: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/cricket_az_adult_naive_plot_combined.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex='collapse' (combining male and female), experience='experienced', age='adolescent'
Filtered out 3 mice with average values > 150 degrees

After filtering: 18 mice/conditions with valid cricket_az data
Values per mouse: count    18.000000
mean      2.111111
std       1.323493
min       1.000000
25%       1.000000
50%       2.000000
75%       2.750000
max       5.000000
Name: cricket_az_count, dtype: float64
Average cricket_az: count    18.000000
mean     -0.023926
std       0.894300
min      -1.909965
25%      -0.354733
50%      -0.235631
75%       0.090923
max       1.784082
Name: cricket_az_mean, dtype: float64
Mice with at least 1 trials: 18

Converted cricket_az_mean from radians to degrees and took absolute values for plotting only

Summary statistics for cricket_az_mean:
count

/tmp/ipykernel_16401/2709766471.py:366: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/cricket_az_adolescent_experienced_plot_combined.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex='collapse' (combining male and female), experience='experienced', age='adult'
Filtered out 1 mice with average values > 150 degrees

After filtering: 18 mice/conditions with valid cricket_az data
Values per mouse: count    18.000000
mean      2.111111
std       1.022620
min       1.000000
25%       1.000000
50%       2.000000
75%       3.000000
max       4.000000
Name: cricket_az_count, dtype: float64
Average cricket_az: count    18.000000
mean      0.103680
std       0.810725
min      -2.307326
25%      -0.084580
50%       0.024561
75%       0.472869
max       1.330145
Name: cricket_az_mean, dtype: float64
Mice with at least 1 trials: 18

Converted cricket_az_mean from radians to degrees and took absolute values for plotting only

Summary statistics for cricket_az_mean:

/tmp/ipykernel_16401/2709766471.py:366: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/cricket_az_adult_experienced_plot_combined.svg


/tmp/ipykernel_16401/2709766471.py:366: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [28]:
variable = 'cricket_az'
for experience in ['naive', 'experienced']:
    for age in ['adolescent', 'adult']:
        _ = create_scatter_plot(csv_file, 
            variable, 
            sex= ['male', 'female'], 
            experience=experience, 
            age=age, 
            hue_variable='freeze',
            show_plot=True, 
            font_sz=8, 
            axis_fontsize=8,
            plot_per_mouse_average=True,
            vmin = 0,
            vmax = 150,
            min_trials=1,
            circular_variable=True,
            output_file=os.path.join(SVG_DIR, f'{variable}_{age}_{experience}_plot.svg'),
            save_svg=True)

Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='naive', age='adolescent'
Filtered out 2 mice with average values > 150 degrees

After filtering: 17 mice/conditions with valid cricket_az data
Values per mouse: count    17.000000
mean      1.411765
std       0.618347
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       3.000000
Name: cricket_az_count, dtype: float64
Average cricket_az: count    17.000000
mean      0.076029
std       0.888442
min      -1.914417
25%      -0.288170
50%       0.113696
75%       0.494945
max       1.554783
Name: cricket_az_mean, dtype: float64
Mice with at least 1 trials: 17

Converted cricket_az_mean from radians to degrees and took absolute values for plotting only

Summary statistics for cricket_az_mean:
count     17.000000
mean      38.567370
std       32.108679
min        0.186819
25%       13.642732
50%       28.358264
75%       56.529042
max      109.688020
Nam

Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/cricket_az_adolescent_naive_plot.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='naive', age='adult'

After filtering: 23 mice/conditions with valid cricket_az data
Values per mouse: count    23.000000
mean      1.347826
std       0.572768
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       3.000000
Name: cricket_az_count, dtype: float64
Average cricket_az: count    23.000000
mean     -0.250531
std       1.174422
min      -2.034846
25%      -0.959414
50%      -0.353420
75%       0.418228
max       2.289274
Name: cricket_az_mean, dtype: float64
Mice with at least 1 trials: 23

Converted cricket_az_mean from radians to degrees and took absolute values for plotting only

Summary statistics for cricket_az_mean:
count     23.000000
mean      54.704970
std       40.182300
min        4.442775
25%       2

/tmp/ipykernel_16401/2709766471.py:366: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/cricket_az_adult_naive_plot.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='experienced', age='adolescent'

After filtering: 22 mice/conditions with valid cricket_az data
Values per mouse: count    22.000000
mean      1.863636
std       0.888844
min       1.000000
25%       1.000000
50%       2.000000
75%       2.000000
max       4.000000
Name: cricket_az_count, dtype: float64
Average cricket_az: count    22.000000
mean     -0.221359
std       1.000020
min      -2.519477
25%      -0.443965
50%      -0.211687
75%       0.090923
max       1.784082
Name: cricket_az_mean, dtype: float64
Mice with at least 1 trials: 22

Converted cricket_az_mean from radians to degrees and took absolute values for plotting only
Filtered out 1 data points with values > 150 degrees

Summary statistics for cricket_az_mean:
count     21.000000
mean      34

/tmp/ipykernel_16401/2709766471.py:366: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/cricket_az_adolescent_experienced_plot.svg
Dataset loaded: 137 rows, 14 columns
Total mice: 67
Filters applied: sex in ['male', 'female'], experience='experienced', age='adult'
Filtered out 1 mice with average values > 150 degrees

After filtering: 20 mice/conditions with valid cricket_az data
Values per mouse: count    20.000000
mean      1.900000
std       0.718185
min       1.000000
25%       1.750000
50%       2.000000
75%       2.000000
max       4.000000
Name: cricket_az_count, dtype: float64
Average cricket_az: count    20.000000
mean      0.096265
std       0.785035
min      -2.307326
25%      -0.193731
50%       0.343902
75%       0.490633
max       1.330145
Name: cricket_az_mean, dtype: float64
Mice with at least 1 trials: 20

Converted cricket_az_mean from radians to degrees and took absolute values for plotting only

Summary statistics for cricket_az_mean:
count     20.000000
mean 

/tmp/ipykernel_16401/2709766471.py:366: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Figure saved to /home/arnab/Code/hoylab_repos/hoylab_analysis/virtualcricket_loom/figure_svgs/cricket_az_adult_experienced_plot.svg


/tmp/ipykernel_16401/2709766471.py:366: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Cricket Azimuth Distribution (Polar)

In [29]:
result = pd.read_csv(csv_file)
gp1 = np.argwhere((result.experience == 'experienced'))
gp2 = np.argwhere((result.experience == 'naive'))

experienced_az = result.cricket_az.iloc[gp1.flatten()]
naive_az = result.cricket_az.iloc[gp2.flatten()]

experienced_clean = experienced_az.dropna()
naive_clean = naive_az.dropna()

experienced_filtered = experienced_clean[
    ~((experienced_clean >= 2.5) | (experienced_clean <= -2.5))
]

naive_filtered = naive_clean[
    ~((naive_clean >= 2.5) | (naive_clean <= -2.5))
]

fig, axes = plt.subplots(1, 1, figsize=(12, 10), subplot_kw=dict(projection='polar'))

n_bins = np.linspace(-np.pi, np.pi, 30)

axes.hist(experienced_filtered, bins=n_bins, alpha=0.3, density=True, edgecolor='black', linewidth=1.5)
axes.hist(naive_filtered, bins=n_bins, alpha=0.3, color='orange', density=True, edgecolor='black', linewidth=1.5)
axes.set_theta_zero_location('N')
axes.set_theta_direction(-1)
axes.tick_params(labelsize=40)

sector_start = np.deg2rad(-20)
sector_end = np.deg2rad(20)

theta_sector = np.linspace(sector_start, sector_end, 100)
r_max = axes.get_ylim()[1]
axes.fill_between(theta_sector, 0, r_max, alpha=0.2, color='gray', label='\u00b120\u00b0 sector')

plt.savefig(os.path.join(SVG_DIR, 'cricket_az_distribution.svg'), format='svg', dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

/tmp/ipykernel_16401/611418257.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
